# Locus.ai — Baseline vs Fine-tuned Benchmark
**Model:** Qwen/Qwen3.5-2B (baseline) vs `merged_model/` (fine-tuned, merged 16-bit)

Benchmarks:
1. JSON Parse Rate
2. Primary Location Exact-Match Accuracy
3. ROUGE-L on full response
4. Latency (tokens/sec)
5. Summary bar chart

In [3]:
# ── Install / upgrade dependencies ────────────────────────────────────────────
import importlib.util, os

!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install -qqq rouge-score

Using Python 3.12.12 environment at: /usr
Resolved 4 packages in 14ms                                          
Checked 4 packages in 0.32ms
Using Python 3.12.12 environment at: /usr
Checked 1 package in 92ms


In [2]:
# ── Shared imports ─────────────────────────────────────────────────────────────
import json, re, time, gc
import torch
import numpy as np
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 512
NUM_SAMPLES    = 100   # set to None to run the full test set
TEST_FILE      = "data/splits/test.jsonl"

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## 1 — Load Models

In [5]:
# ── Inspect what's actually saved in Drive ────────────────────────────────────
import os

DRIVE_MODEL_PATH = "/content/drive/MyDrive/locus.ai/merged_model"

print(f"Contents of {DRIVE_MODEL_PATH}:\n")
for f in sorted(os.listdir(DRIVE_MODEL_PATH)):
    size_mb = os.path.getsize(os.path.join(DRIVE_MODEL_PATH, f)) / 1e6
    print(f"  {f:<60}  {size_mb:>8.1f} MB")

Contents of /content/drive/MyDrive/locus.ai/merged_model:



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/locus.ai/merged_model'

In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

MessageError: User cancelled dfs_ephemeral authorization

In [ ]:
drive.flush_and_unmount()
print("Drive unmounted.")

In [8]:
# ── Upload merged_model from local machine to Colab runtime ───────────────────
# 1. On your LOCAL machine, zip the folder first:
#      zip -r merged_model.zip merged_model/
# 2. Run this cell — a file picker will appear; select merged_model.zip

import os, zipfile
from google.colab import files

print("Select merged_model.zip to upload...")
uploaded = files.upload()   # opens browser file picker

zip_name = next(iter(uploaded))
dest_dir = "/content/merged_model"

print(f"\nExtracting {zip_name} → {dest_dir} ...")
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("/content")

# Handle the common case where zip contains a top-level 'merged_model/' folder
if not os.path.isdir(dest_dir):
    # zip may have been created from inside the folder — rewrap
    extracted = [d for d in os.listdir("/content") if os.path.isdir(f"/content/{d}") and "merged" in d.lower()]
    print(f"Could not find {dest_dir}. Extracted dirs: {extracted}")
else:
    files_list = sorted(os.listdir(dest_dir))
    total_mb   = sum(os.path.getsize(os.path.join(dest_dir, f)) for f in files_list) / 1e6
    print(f"\nReady: {dest_dir}  ({total_mb:.0f} MB total)")
    for f in files_list:
        mb = os.path.getsize(os.path.join(dest_dir, f)) / 1e6
        print(f"  {f:<60}  {mb:>8.1f} MB")

Select merged_model.zip to upload...


KeyboardInterrupt: 

In [4]:
# ── Baseline: vanilla Qwen3.5-2B ──────────────────────────────────────────────
print("Loading BASELINE model...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "Qwen/Qwen3.5-2B",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = False,
    load_in_16bit  = True,
)
FastLanguageModel.for_inference(base_model)
print("Baseline model ready.")

Loading BASELINE model...
==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Baseline model ready.


In [ ]:
# ── Fine-tuned: Step 1 — Load merged model (local copy), convert to GGUF ──────
DRIVE_MODEL_PATH = "/content/merged_model"   # downloaded from Drive above
GGUF_DIR         = "ft_gguf"
GGUF_Q4_PATH     = f"{GGUF_DIR}/model-unsloth.Q4_K_M.gguf"

print(f"Loading merged model from {DRIVE_MODEL_PATH} ...")
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = DRIVE_MODEL_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = False,
    load_in_16bit  = True,
)

print("Converting to GGUF Q4_K_M (this may take a few minutes)...")
ft_model.save_pretrained_gguf(GGUF_DIR, ft_tokenizer, quantization_method="q4_k_m")

import os
size_gb = os.path.getsize(GGUF_Q4_PATH) / 1e9
print(f"\nGGUF saved: {GGUF_Q4_PATH}  ({size_gb:.2f} GB)")

del ft_model
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("HF model unloaded from memory.")

In [ ]:
# ── Fine-tuned: Step 2 — Install llama-cpp-python (CUDA) + load GGUF ──────────
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python -qqq --upgrade --force-reinstall

from llama_cpp import Llama

ft_llm = Llama(
    model_path   = GGUF_Q4_PATH,
    n_gpu_layers = -1,          # offload all layers to GPU
    n_ctx        = MAX_SEQ_LENGTH,
    verbose      = False,
)
print(f"Fine-tuned GGUF model loaded: {GGUF_Q4_PATH}")

In [ ]:
# ── Fine-tuned: inference helper (llama-cpp-python) ───────────────────────────
import time

def run_inference_gguf(llm, tok, examples, max_new=MAX_NEW_TOKENS):
    """Greedy inference via llama-cpp-python; returns (predictions, tokens_per_sec list)."""
    preds, tps_list = [], []
    for ex in examples:
        messages = [{"role": "user", "content": ex["instruction"]}]
        prompt   = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        t0  = time.perf_counter()
        out = llm(
            prompt,
            max_tokens  = max_new,
            temperature = 0.0,   # greedy
            echo        = False,
        )
        elapsed = time.perf_counter() - t0

        text  = out["choices"][0]["text"].strip()
        n_gen = out["usage"]["completion_tokens"]
        tps_list.append(n_gen / elapsed if elapsed > 0 else 0)
        preds.append(text)

    return preds, tps_list

print("GGUF inference helper defined.")

## 2 — Test Data & Helpers

In [ ]:
# ── Load test examples ────────────────────────────────────────────────────────
def load_test(path, n=None):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows[:n] if n else rows

test_examples = load_test(TEST_FILE, NUM_SAMPLES)
refs          = [ex["response"] for ex in test_examples]
print(f"Loaded {len(test_examples)} test examples.")

In [ ]:
# ── Inference runner ──────────────────────────────────────────────────────────
def run_inference(mdl, tok, examples, max_new=MAX_NEW_TOKENS):
    """Run greedy inference; returns (predictions, tokens_per_sec list)."""
    mdl.eval()
    preds, tps_list = [], []
    for ex in examples:
        messages = [{"role": "user", "content": ex["instruction"]}]
        prompt   = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs   = tok(prompt, return_tensors="pt").to(mdl.device)
        n_input  = inputs["input_ids"].shape[-1]

        t0 = time.perf_counter()
        with torch.no_grad():
            out = mdl.generate(
                **inputs,
                max_new_tokens = max_new,
                do_sample      = False,
                temperature    = 1.0,
                use_cache      = True,
            )
        elapsed = time.perf_counter() - t0

        n_generated = out.shape[-1] - n_input
        tps_list.append(n_generated / elapsed if elapsed > 0 else 0)

        decoded = tok.decode(out[0][n_input:], skip_special_tokens=True)
        preds.append(decoded.strip())

    return preds, tps_list


# ── Metric helpers ────────────────────────────────────────────────────────────
def extract_primary(text):
    """Parse primary_location from JSON response, with regex fallback."""
    try:
        obj = json.loads(text)
        return str(obj.get("primary_location", "")).strip().lower()
    except Exception:
        m = re.search(r'"primary_location"\s*:\s*"([^"]+)"', text)
        return m.group(1).strip().lower() if m else ""


def rouge_l_f1(pred, ref):
    """Token-level ROUGE-L F1."""
    p_tok, r_tok = pred.split(), ref.split()
    if not p_tok or not r_tok:
        return 0.0
    m, n = len(p_tok), len(r_tok)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if p_tok[i-1] == r_tok[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs  = dp[m][n]
    prec = lcs / m
    rec  = lcs / n
    return 2 * prec * rec / (prec + rec) if prec + rec else 0.0


def compute_metrics(preds, refs, tps_list):
    json_ok, loc_exact, rl_scores = [], [], []
    for pred, ref in zip(preds, refs):
        try:
            json.loads(pred)
            json_ok.append(1)
        except Exception:
            json_ok.append(0)
        loc_exact.append(int(extract_primary(pred) == extract_primary(ref)))
        rl_scores.append(rouge_l_f1(pred, ref))
    return {
        "json_parse_rate (%)":    np.mean(json_ok)    * 100,
        "location_accuracy (%)":  np.mean(loc_exact)  * 100,
        "rouge_l (%)": np.mean(rl_scores)  * 100,
        "avg_tokens_per_sec":     np.mean(tps_list),
    }

print("Helpers defined.")

## 3 — Run Inference

In [ ]:
# ── Baseline inference ────────────────────────────────────────────────────────
print(f"Running baseline inference on {len(test_examples)} examples...")
baseline_preds, baseline_tps = run_inference(base_model, base_tokenizer, test_examples)
print(f"Done. Avg speed: {np.mean(baseline_tps):.1f} tok/s")

In [ ]:
# ── Fine-tuned inference (GGUF via llama-cpp-python) ─────────────────────────
# Uses base_tokenizer to apply the chat template (identical vocab/template)
print(f"Running fine-tuned GGUF inference on {len(test_examples)} examples...")
ft_preds, ft_tps = run_inference_gguf(ft_llm, base_tokenizer, test_examples)
print(f"Done. Avg speed: {np.mean(ft_tps):.1f} tok/s")

## 4 — Benchmark 1: JSON Parse Rate
Measures what fraction of responses are valid JSON — critical since downstream code parses the output.

In [ ]:
import matplotlib.pyplot as plt

def json_parse_rate(preds):
    results = []
    for pred in preds:
        try:
            json.loads(pred)
            results.append(1)
        except Exception:
            results.append(0)
    return results

base_json = json_parse_rate(baseline_preds)
ft_json   = json_parse_rate(ft_preds)

base_rate = np.mean(base_json) * 100
ft_rate   = np.mean(ft_json)   * 100

print(f"Baseline  JSON Parse Rate: {base_rate:.1f}%")
print(f"Fine-tuned JSON Parse Rate: {ft_rate:.1f}%")
print(f"Delta: {ft_rate - base_rate:+.1f}pp")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Baseline", "Fine-tuned"], [base_rate, ft_rate],
       color=["salmon", "steelblue"], width=0.4)
ax.set_ylabel("JSON Parse Rate (%)")
ax.set_title("Benchmark 1: JSON Parse Rate")
ax.set_ylim(0, 105)
for i, v in enumerate([base_rate, ft_rate]):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("outputs_qwen35/bench_json_parse.png", dpi=150)
plt.show()

## 5 — Benchmark 2: Primary Location Exact-Match Accuracy
Extracts the `primary_location` field and checks exact string match against the ground-truth.

In [ ]:
base_loc_hits = [int(extract_primary(p) == extract_primary(r)) for p, r in zip(baseline_preds, refs)]
ft_loc_hits   = [int(extract_primary(p) == extract_primary(r)) for p, r in zip(ft_preds,       refs)]

base_loc_acc = np.mean(base_loc_hits) * 100
ft_loc_acc   = np.mean(ft_loc_hits)   * 100

print(f"Baseline   Location Accuracy: {base_loc_acc:.1f}%")
print(f"Fine-tuned Location Accuracy: {ft_loc_acc:.1f}%")
print(f"Delta: {ft_loc_acc - base_loc_acc:+.1f}pp")

# ── Per-location breakdown (fine-tuned) ──
from collections import defaultdict
loc_correct = defaultdict(int)
loc_total   = defaultdict(int)
for p, r, hit in zip(ft_preds, refs, ft_loc_hits):
    true_loc = extract_primary(r)
    loc_total[true_loc]   += 1
    loc_correct[true_loc] += hit

print("\nPer-location accuracy (fine-tuned, top-10 by count):")
sorted_locs = sorted(loc_total.items(), key=lambda x: -x[1])[:10]
for loc, total in sorted_locs:
    acc = loc_correct[loc] / total * 100
    print(f"  {loc:<30} {loc_correct[loc]:>3}/{total:<3}  ({acc:.0f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Baseline", "Fine-tuned"], [base_loc_acc, ft_loc_acc],
       color=["salmon", "steelblue"], width=0.4)
ax.set_ylabel("Location Exact-Match (%)")
ax.set_title("Benchmark 2: Primary Location Accuracy")
ax.set_ylim(0, 105)
for i, v in enumerate([base_loc_acc, ft_loc_acc]):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("outputs_qwen35/bench_location_acc.png", dpi=150)
plt.show()

## 6 — Benchmark 3: ROUGE-L (Full Response)
Token-level ROUGE-L F1 over the complete model output vs. the reference response.

In [ ]:
base_rl = [rouge_l_f1(p, r) for p, r in zip(baseline_preds, refs)]
ft_rl   = [rouge_l_f1(p, r) for p, r in zip(ft_preds,       refs)]

base_rl_mean = np.mean(base_rl) * 100
ft_rl_mean   = np.mean(ft_rl)   * 100

print(f"Baseline   ROUGE-L: {base_rl_mean:.1f}%")
print(f"Fine-tuned ROUGE-L: {ft_rl_mean:.1f}%")
print(f"Delta: {ft_rl_mean - base_rl_mean:+.1f}pp")

# Distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, scores, label, color in [
    (axes[0], base_rl, "Baseline",    "salmon"),
    (axes[1], ft_rl,   "Fine-tuned",  "steelblue"),
]:
    ax.hist([s * 100 for s in scores], bins=20, color=color, edgecolor="white")
    ax.axvline(np.mean(scores) * 100, color="black", linestyle="--", label=f"mean={np.mean(scores)*100:.1f}%")
    ax.set_title(f"{label} — ROUGE-L distribution")
    ax.set_xlabel("ROUGE-L (%)")
    ax.set_ylabel("Count")
    ax.legend()
plt.tight_layout()
plt.savefig("outputs_qwen35/bench_rouge_l.png", dpi=150)
plt.show()

## 7 — Benchmark 4: Inference Latency (tokens/sec)

In [ ]:
base_tps_mean = np.mean(baseline_tps)
ft_tps_mean   = np.mean(ft_tps)

print(f"Baseline   avg tokens/sec: {base_tps_mean:.1f}")
print(f"Fine-tuned avg tokens/sec: {ft_tps_mean:.1f}")
print(f"Delta: {ft_tps_mean - base_tps_mean:+.1f} tok/s")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, tps, label, color in [
    (axes[0], baseline_tps, "Baseline",   "salmon"),
    (axes[1], ft_tps,       "Fine-tuned", "steelblue"),
]:
    ax.hist(tps, bins=20, color=color, edgecolor="white")
    ax.axvline(np.mean(tps), color="black", linestyle="--", label=f"mean={np.mean(tps):.1f}")
    ax.set_title(f"{label} — Tokens/sec distribution")
    ax.set_xlabel("Tokens/sec")
    ax.set_ylabel("Count")
    ax.legend()
plt.tight_layout()
plt.savefig("outputs_qwen35/bench_latency.png", dpi=150)
plt.show()

## 8 — Summary: All Metrics Side-by-Side

In [ ]:
base_metrics = compute_metrics(baseline_preds, refs, baseline_tps)
ft_metrics   = compute_metrics(ft_preds,       refs, ft_tps)

# ── Print table ───────────────────────────────────────────────────────────────
print(f"{'Metric':<28} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 62)
for k in base_metrics:
    b, f = base_metrics[k], ft_metrics[k]
    unit = " tok/s" if "tok" in k else "%"
    print(f"{k:<28} {b:>9.1f}{unit}  {f:>9.1f}{unit}  {f-b:>+7.1f}")

# ── Bar chart for % metrics ───────────────────────────────────────────────────
pct_keys   = [k for k in base_metrics if "%" in k]
labels     = [k.replace(" (%)", "").replace("_", " ") for k in pct_keys]
base_vals  = [base_metrics[k] for k in pct_keys]
ft_vals    = [ft_metrics[k]   for k in pct_keys]

x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars_b = ax.bar(x - w/2, base_vals, w, label="Baseline",    color="salmon")
bars_f = ax.bar(x + w/2, ft_vals,   w, label="Fine-tuned",  color="steelblue")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("Score (%)")
ax.set_title("Locus.ai — Baseline vs Fine-tuned: All Benchmarks")
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
for bar in list(bars_b) + list(bars_f):
    ax.annotate(f"{bar.get_height():.1f}%",
                (bar.get_x() + bar.get_width() / 2, bar.get_height() + 1),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("outputs_qwen35/benchmark_summary.png", dpi=150)
plt.show()
print("\nSaved: outputs_qwen35/benchmark_summary.png")

## 9 — Qualitative Inspection: Side-by-Side Predictions

In [ ]:
# ── Print N side-by-side examples ─────────────────────────────────────────────
N_SHOW = 5

for i in range(N_SHOW):
    print(f"{'='*70}")
    print(f"Example {i+1}")
    print(f"INSTRUCTION: {test_examples[i]['instruction'][:200]}")
    print(f"\nREFERENCE:   {refs[i]}")
    print(f"\nBASELINE:    {baseline_preds[i][:300]}")
    print(f"\nFINE-TUNED:  {ft_preds[i][:300]}")
    
    b_loc = extract_primary(baseline_preds[i])
    f_loc = extract_primary(ft_preds[i])
    r_loc = extract_primary(refs[i])
    print(f"\nLocation — ref: '{r_loc}'  |  baseline: '{b_loc}'  |  ft: '{f_loc}'")
    print()